In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

In [ ]:
def load_dataset(folder_path):

    folder = Path(folder_path)
    csv_path = folder / "_classes.csv"

    df = pd.read_csv(csv_path)

    images = []
    labels = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Loading {folder.name}"):
        filename = row.iloc[0]  # First column is filename
        img_path = folder / filename

        if not img_path.exists():
            print(f"Warning: {img_path} not found, skipping")
            continue

        # Load image and convert to grayscale
        img = Image.open(img_path).convert("L")  # 'L' mode is grayscale
        img_array = np.array(img)
        images.append(img_array)

        # Find which score column has the 1 (get the label 0-5)
        # Skip column 0 (filename) and column 1 (Unlabeled)
        score_values = row.iloc[2:8].values  # Columns for scores 0-5
        label = np.argmax(score_values)
        labels.append(label)

    X = np.array(images)
    y = np.array(labels)

    return X, y

In [3]:
base = Path("./data/processed")

print("Loading training data...")
X_train, y_train = load_dataset(base / "train")

print("Loading validation data...")
X_val, y_val = load_dataset(base / "valid")

print("Loading test data...")
X_test, y_test = load_dataset(base / "test")

downsample_factor = 2
X_train = X_train[:, ::downsample_factor, ::downsample_factor]
X_val = X_val[:, ::downsample_factor, ::downsample_factor]
X_test = X_test[:, ::downsample_factor, ::downsample_factor]

Loading training data...


Loading train: 100%|██████████| 20602/20602 [00:08<00:00, 2430.08it/s]


Loading validation data...


Loading valid: 100%|██████████| 2168/2168 [00:00<00:00, 2486.02it/s]


Loading test data...


Loading test: 100%|██████████| 1412/1412 [00:00<00:00, 2254.72it/s]


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

# Convert to tensors and add channel dimension
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)  # (N, 1, H, W)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
y_val_t = torch.LongTensor(y_val)
y_test_t = torch.LongTensor(y_test)

batch_size = 64

# DataLoaders
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size)


# CNN
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


num_classes = len(torch.unique(y_train_t))
model = CNN(num_classes)
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")
model = model.to(device)

class_counts = np.bincount(y_train, minlength=num_classes)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes  # normalize
class_weights_t = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def compute_auc(loader, model, device, num_classes):
    """Compute multiclass AUC (OVR macro) using softmax probabilities."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            probs = torch.softmax(model(X_batch), dim=1)
            all_probs.append(probs.cpu())
            all_labels.append(y_batch)
    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")


# Training loop
for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    # Validation AUC (multiclass OVR macro)
    val_auc = compute_auc(val_loader, model, device, num_classes)
    print(f"Epoch {epoch+1}: Val AUC = {val_auc:.4f}")

# Test AUC
test_auc = compute_auc(test_loader, model, device, num_classes)
print(f"\nTest AUC: {test_auc:.4f}")

Epoch 1: Val AUC = 0.5866
Epoch 2: Val AUC = 0.5808
Epoch 3: Val AUC = 0.5813
Epoch 4: Val AUC = 0.5961
Epoch 5: Val AUC = 0.7233
Epoch 6: Val AUC = 0.7736
Epoch 7: Val AUC = 0.7851
Epoch 8: Val AUC = 0.7743
Epoch 9: Val AUC = 0.8085
Epoch 10: Val AUC = 0.8328
Epoch 11: Val AUC = 0.8289
Epoch 12: Val AUC = 0.8240
Epoch 13: Val AUC = 0.8394
Epoch 14: Val AUC = 0.8408
Epoch 15: Val AUC = 0.8402
Epoch 16: Val AUC = 0.8286
Epoch 17: Val AUC = 0.8551
Epoch 18: Val AUC = 0.8375
Epoch 19: Val AUC = 0.8471
Epoch 20: Val AUC = 0.8488

Test AUC: 0.8625


In [ ]:
# Collect softmax probs and hard predictions
all_probs, all_preds, all_labels = [], [], []
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.softmax(logits, dim=1).cpu()
        preds = logits.argmax(1).cpu()
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(y_batch)

all_probs = torch.cat(all_probs).numpy()
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Binarize: scores 0,1,2 -> affected (1), scores 3,4,5 -> not affected (0)
binary_preds = (all_preds <= 2).astype(int)
binary_labels = (all_labels <= 2).astype(int)

TN = ((binary_labels == 0) & (binary_preds == 0)).sum()
FP = ((binary_labels == 0) & (binary_preds == 1)).sum()
FN = ((binary_labels == 1) & (binary_preds == 0)).sum()
TP = ((binary_labels == 1) & (binary_preds == 1)).sum()

print([[int(TN), int(FP)], [int(FN), int(TP)]])

TN_norm = int(TN) / (int(TN) + int(FP)) * 100
FP_norm = int(FP) / (int(TN) + int(FP)) * 100
FN_norm = int(FN) / (int(FN) + int(TP)) * 100
TP_norm = int(TP) / (int(FN) + int(TP)) * 100

print([[TN_norm, FP_norm], [FN_norm, TP_norm]])

# Multiclass AUC (OVR macro) across all 6 score classes
auc_multiclass = roc_auc_score(
    all_labels, all_probs, multi_class="ovr", average="macro"
)
print(f"\nTest AUC (multiclass OVR macro): {auc_multiclass:.4f}")

# Binary AUC after binarizing predictions (affected vs not affected)
binary_probs = all_probs[:, :3].sum(axis=1)  # P(score in 0,1,2) = P(affected)
auc_binary = roc_auc_score(binary_labels, binary_probs)
print(f"Test AUC (binary affected vs not): {auc_binary:.4f}")

[[1109, 163], [38, 102]]
[[87.18553459119497, 12.814465408805031], [27.142857142857142, 72.85714285714285]]

Test AUC (multiclass OVR macro): 0.8625
Test AUC (binary affected vs not): 0.9237
